# 01 - Explore the Data

**Business Context:** HydraB Power manufactures electric buses and needs a unified view of their fleet. Today, vehicle data lives in two disconnected systems:
- **Salesforce CRM** - sales pipeline, vehicle master data, delivery tracking, defect reports
- **Odos Telematics API** - real-time GPS location, battery state-of-charge, speed, odometer

The key link between these systems is the **VIN (Vehicle Identification Number)** - a unique chassis code assigned to every bus during manufacturing. By joining on VIN, we can build a complete Vehicle 360 view.

**What this notebook does:**
- Explores the BRONZE data layer (raw ingested data from both systems)
- Validates the CRM tables: Opportunities, Vehicles (Assets), Defect Events, Deliveries
- Parses the nested JSON telemetry from the Odos telematics platform
- Proves the VIN-based join between CRM and telemetry works
- Pulls external weather data from a public API to enrich fleet insights

**Pain point addressed:** These data sources have never been joined before. The sales team cannot see vehicle health, and the operations team cannot see the delivery pipeline. This notebook validates that Snowflake can unify both into a single queryable layer.


In [ ]:
-- Set context
SET user_db = 'HYDRAB_HOL_' || CURRENT_USER();
USE DATABASE IDENTIFIER($user_db);
USE SCHEMA BRONZE;
USE WAREHOUSE HYDRAB_HOL_WH;

## Salesforce: Sales Pipeline

In [ ]:
-- Top opportunities by value
SELECT "Name", "StageName", "Amount", "CloseDate", "Region__c"
FROM OPPORTUNITY
WHERE "Amount" > 1000000
ORDER BY "Amount" DESC
LIMIT 20;

In [ ]:
-- Pipeline by stage
SELECT "StageName", 
       COUNT(*) as deals, 
       SUM("Amount") as total_value,
       ROUND(AVG("Amount"), 0) as avg_deal
FROM OPPORTUNITY
GROUP BY "StageName"
ORDER BY total_value DESC;

## Salesforce: Vehicle Fleet

In [ ]:
-- Fleet composition by model
SELECT "Name" as model,
       COUNT(*) as vehicles,
       COUNT(DISTINCT "AccountId") as customers
FROM ASSET
WHERE "Chassis_Number__c" IS NOT NULL
GROUP BY "Name"
ORDER BY vehicles DESC
LIMIT 15;

## Odos Telemetry: Raw Events

In [ ]:
-- Sample telemetry events (raw JSON)
SELECT *
FROM ODOS_EVENTS
LIMIT 5;

In [ ]:
-- Parse key signals from Odos JSON
-- The data is stored as VARCHAR; we parse it and flatten the signals array
SELECT 
  PARSE_JSON(RAW_JSON):vin::STRING as vin,
  PARSE_JSON(RAW_JSON):startTime::TIMESTAMP as event_time,
  PARSE_JSON(RAW_JSON):vehicleCustomer::STRING as customer,
  sig.value:name::STRING as signal_name,
  sig.value:displayName::STRING as signal_display,
  sig.value:values[0]:value::STRING as first_reading
FROM ODOS_EVENTS,
  LATERAL FLATTEN(input => PARSE_JSON(RAW_JSON):signals) sig
LIMIT 20;

## The VIN Join: Connecting CRM to Telemetry

In [ ]:
-- How many VINs overlap between Salesforce and Odos?
SELECT COUNT(DISTINCT a."Chassis_Number__c") as matching_vins
FROM ASSET a
INNER JOIN (
    SELECT DISTINCT PARSE_JSON("RAW_JSON"):vin::STRING as vin
    FROM ODOS_EVENTS
) o ON a."Chassis_Number__c" = o.vin;

In [ ]:
-- Joined view: vehicle details + latest telemetry location
SELECT
    a."Chassis_Number__c" as vin,
    a."Name" as model,
    a."Status" as status,
    a."LatestLocation__Latitude__s" as lat,
    a."LatestLocation__Longitude__s" as lon,
    a."LatestMileage__c" as mileage
FROM ASSET a
WHERE a."Chassis_Number__c" IS NOT NULL
    AND a."LatestLocation__Latitude__s" IS NOT NULL
LIMIT 20;

## External Data Enrichment: Public API

**Business Context:** HydraB operates buses across the UK and Ireland. Weather directly impacts battery performance (cold = reduced range) and route planning. By pulling publicly available weather data into Snowflake, we can correlate telemetry anomalies with weather events.

This demonstrates Snowflake's ability to combine internal operational data with external public datasets — a key requirement raised by the team.


In [ ]:
# Pull weather data from Open-Meteo (free, no API key required)
# This enriches our fleet data with weather conditions at depot locations
import requests
import json
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import Row

session = get_active_session()

# Key HydraB depot locations (Belfast, London, Dublin)
depots = [
    {"name": "Belfast", "lat": 54.60, "lon": -5.93},
    {"name": "London", "lat": 51.51, "lon": -0.13},
    {"name": "Dublin", "lat": 53.35, "lon": -6.26},
]

# Fetch current weather for each depot
weather_rows = []
for depot in depots:
    url = f"https://api.open-meteo.com/v1/forecast?latitude={depot['lat']}&longitude={depot['lon']}&current=temperature_2m,wind_speed_10m,precipitation&timezone=Europe/London"
    resp = requests.get(url)
    data = resp.json()
    current = data.get('current', {})
    weather_rows.append({
        "DEPOT_NAME": depot['name'],
        "LATITUDE": depot['lat'],
        "LONGITUDE": depot['lon'],
        "TEMPERATURE_C": current.get('temperature_2m'),
        "WIND_SPEED_KMH": current.get('wind_speed_10m'),
        "PRECIPITATION_MM": current.get('precipitation'),
        "FETCHED_AT": data.get('current', {}).get('time')
    })

# Store in Snowflake for the React dashboard to consume
df = session.create_dataframe([Row(**r) for r in weather_rows])
df.write.mode('overwrite').save_as_table('BRONZE.DEPOT_WEATHER')
df.show()


In [ ]:
-- Verify the weather data was stored
-- This table is now available for the React dashboard to display depot conditions
SET user_db = 'HYDRAB_HOL_' || CURRENT_USER();
USE DATABASE IDENTIFIER($user_db);
SELECT * FROM BRONZE.DEPOT_WEATHER;

## Explore Complete!

Key findings:
- 2,492 VINs match between Salesforce and Odos
- Telemetry includes SOC, speed, temperature, GPS coordinates
- Sales pipeline has 1,056 opportunities worth £2.5B+

**Next:** Open `03_build_gold.ipynb` to build the analytics layer.